# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score


## Model Choice

For the baseline model, we selected **Linear Regression**.

The task is a regression problem with time-dependent sales data. Linear regression provides a transparent and interpretable benchmark to evaluate whether engineered time-series features (lags, rolling statistics, seasonality and calendar variables) can sufficiently explain sales variability.

Since demand patterns differ across product groups, we train one separate linear model per Warengruppe. This allows us to capture heterogeneous demand structures and compare predictive strength across product categories.

This baseline serves as a reference point for later comparison with more complex models such as neural networks.


## Feature Selection

The following feature groups were constructed for the baseline model:

**Calendar Features**
- Month
- Weekday (one-hot encoded)
- Weekend indicator
- Public holiday
- School holiday
- Kieler Woche event flag

**Seasonality Encoding**
- Sine and cosine transformation of day-of-year

**Lag Features**
- lag_1, lag_2, lag_7, lag_14

**Rolling Statistics**
- Rolling mean and rolling standard deviation over 7, 14 and 30 days

All lag and rolling features were shifted appropriately to prevent data leakage.


In [ ]:
# Load pre-defined time-based splits
train = pd.read_csv("train_split_merged_expanded_data.csv")
val   = pd.read_csv("val_split_merged_expanded_data.csv")

# Ensure correct datetime format
train["date"] = pd.to_datetime(train["date"])
val["date"]   = pd.to_datetime(val["date"])


## Implementation

The baseline model is implemented using a Scikit-Learn pipeline:

- Mean imputation for missing lag and rolling features
- Linear Regression as estimator
- Separate model per Warengruppe
- Strict time-based train/validation split

Model performance is evaluated on the validation set.


In [ ]:
def adjusted_r2(r2: float, n: int, p: int) -> float:
    if n <= p + 1:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

feature_cols = [col for col in train.columns if col not in ["umsatz", "date"]]
target_col = "umsatz"

results = []
product_groups = sorted(train["warengruppe"].unique())

for wg in product_groups:
    train_wg = train[train["warengruppe"] == wg]
    val_wg   = val[val["warengruppe"] == wg]

    X_train = train_wg[feature_cols]
    y_train = train_wg[target_col]

    X_val = val_wg[feature_cols]
    y_val = val_wg[target_col]

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("reg", LinearRegression())
    ])

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_val_pred   = model.predict(X_val)

    r2_train = r2_score(y_train, y_train_pred)
    r2_val   = r2_score(y_val, y_val_pred)

    adj_r2_train = adjusted_r2(r2_train, len(y_train), len(feature_cols))

    results.append({
        "Warengruppe": wg,
        "R2_train": r2_train,
        "Adj_R2_train": adj_r2_train,
        "R2_val": r2_val
    })

results_df = pd.DataFrame(results)
results_df


## Evaluation

Model performance was evaluated using:

- **R² (Coefficient of Determination)**
- **Adjusted R² (training data)**

Validation R² per Warengruppe:

- WG1: 0.476
- WG2: 0.862
- WG3: 0.848
- WG4: 0.182
- WG5: 0.248
- WG6: 0.552

The model performs particularly well for Warengruppe 2 and 3, indicating stable and predictable demand patterns. Warengruppe 1 and 6 show moderate predictability, while 4 and 5 exhibit weaker performance, likely due to higher noise and structural variability.
